In [44]:
import sys
sys.path.append("..")
from models.MIL import RadiomicsMIL
from dataloaders.deep_dataloaders import get_dataloaders_deep_learning
from utils import read_yaml_file, test_model, compute_classification_metrics, save_json, test_model_graph
import torch
import numpy as np
import os
from glob import glob
from pathlib import Path

In [7]:
data_root = "../../new_dataset/"
use_tda = False
fold_index = 0
model_configs = read_yaml_file("../configs/mil.yaml")
model = RadiomicsMIL.load_from_checkpoint("/home/reza/Documents/Reza_projects/08_drarabi_lymphnodes/cleaned_code/Results/mil_radiomics/fold_2/checkpoints/best.ckpt", 
                        input_dim=110,
                        config=model_configs)
train_loader, val_loader, test_loader = get_dataloaders_deep_learning(data_root, use_tda, batch_size=model_configs['batch_size'], fold_index=fold_index)

Train size: 102, Val size: 18, Test size: 31
No valid lesions found for KOLIVAND HASTI 96043105, skipping.
No valid lesions found for DEHGHAN NAYERI MARZIYEH BEYGOM 9484, skipping.


In [9]:
y_true, y_pred, y_prob = test_model(model, test_loader)
compute_classification_metrics('mil', y_true, y_pred, y_prob)

{'model': 'mil',
 'accuracy': 0.8709677419354839,
 'precision': 0.8823529411764706,
 'recall': 0.8823529411764706,
 'specificity': np.float64(0.8571428571428571),
 'f1_score': 0.8823529411764706,
 'roc_auc': 0.9327731092436975}

In [12]:
def get_attention_preds(model, test_loader):
    all_preds = []
    all_probs = []
    all_labels = []
    attention_masks = []
    model.eval()
    with torch.no_grad():
        for batch in test_loader:
            x, y, mask = batch['features'], batch['labels'], batch.get('pad_mask', None)
            logits, attention = model.get_pred_and_attention(x.to(model.device), mask.to(model.device) if mask is not None else None)
            probs = torch.softmax(logits, dim=1)[:, 1]
            preds = torch.argmax(logits, dim=1)
            all_preds.extend(preds.cpu().numpy())
            all_probs.extend(probs.cpu().numpy())
            all_labels.extend(y.cpu().numpy())
            attention_masks.extend(attention.cpu().numpy())
    return np.array(all_labels), np.array(all_preds), np.array(all_probs), np.array(attention_masks)

In [13]:
y_true, y_pred, y_prob, attention_masks = get_attention_preds(model, test_loader)

In [35]:
for patient_id in range(attention_masks.shape[0]):
    nodes_importance = attention_masks[patient_id]
    nodes_coordinates = test_loader.dataset.aggregated_data[patient_id]['nodes_coordinates']
    print("node importances for patient ", patient_id)
    print(f"len nodes importance: {len(nodes_importance[nodes_importance>0])}, len total nodes: {len(nodes_coordinates)}")
    print(nodes_importance[nodes_importance>0])
    print("nodes coordinates:")
    print(nodes_coordinates)
    print("="*20)

node importances for patient  0
len nodes importance: 5, len total nodes: 5
[0.1807706  0.16623965 0.2173489  0.20799567 0.22764513]
nodes coordinates:
['(109.16666666666667, 101.39115646258503, 184.27551020408163)', '(116.17693836978131, 108.8469184890656, 183.55864811133202)', '(84.10546875, 94.4140625, 253.81640625)', '(94.97889182058047, 96.17238346525946, 256.6772207563764)', '(99.98009950248756, 102.70149253731343, 254.91044776119404)']
node importances for patient  1
len nodes importance: 3, len total nodes: 3
[0.31256282 0.35315302 0.33428416]
nodes coordinates:
['(129.32389162561577, 106.55788177339902, 257.67056650246303)', '(138.38709677419354, 95.24193548387096, 254.2741935483871)', '(134.60849056603774, 103.65094339622641, 263.9481132075472)']
node importances for patient  2
len nodes importance: 1, len total nodes: 1
[1.]
nodes coordinates:
['(138.56923076923076, 112.73846153846154, 282.2846153846154)']
node importances for patient  3
len nodes importance: 19, len total n

In [42]:
import SimpleITK as sitk

def read_sitk_image(file_path):
    img = sitk.ReadImage(file_path)
    return img

def sitk_image_to_array(sitk_image):
    array = sitk.GetArrayFromImage(sitk_image)
    return array

def array_to_sitk_image(array, reference_sitk_image):
    sitk_image = sitk.GetImageFromArray(array)
    sitk_image.CopyInformation(reference_sitk_image)
    return sitk_image

In [55]:
patient_id = 23

nodes_importance = attention_masks[patient_id]
nodes_coordinates = test_loader.dataset.aggregated_data[patient_id]['nodes_coordinates']
print("node importances for patient ", patient_id)
print(f"len nodes importance: {len(nodes_importance[nodes_importance>0])}, len total nodes: {len(nodes_coordinates)}")
print(nodes_importance[nodes_importance>0])
print("nodes coordinates:")
print(nodes_coordinates)


node importances for patient  23
len nodes importance: 12, len total nodes: 12
[0.08948013 0.09031311 0.09047768 0.08774967 0.10980751 0.07049412
 0.0786042  0.08069063 0.08900055 0.07224417 0.07233895 0.06879935]
nodes coordinates:
['(104.82025677603424, 128.94293865905848, 245.29671897289586)', '(121.09655172413792, 129.03275862068966, 252.4948275862069)', '(87.69568921451696, 118.95723291872551, 238.914636224229)', '(129.42187860604662, 119.6912070159243, 240.28456035079623)', '(92.36720554272517, 133.60739030023095, 236.78983833718246)', '(122.99860178970917, 118.7841163310962, 226.6006711409396)', '(88.18106995884774, 117.14814814814815, 219.79835390946502)', '(91.37666174298376, 119.57311669128508, 225.48818316100443)', '(111.90310559006211, 111.09937888198758, 218.12670807453415)', '(124.03418803418803, 116.76068376068376, 214.89743589743588)', '(114.42708333333333, 106.30208333333333, 213.04166666666666)', '(120.06666666666666, 108.60606060606061, 214.3212121212121)']


In [56]:
pid = test_loader.dataset.aggregated_data[patient_id]['ID']
data_path = Path(data_root) / "Masih-SUV" / f"{pid}" 
img_path = glob(str(data_path / "*_prep_img.nii.gz"))
seg_path = glob(str(data_path / "*_prep_seg.nii.gz"))
img_sitk = read_sitk_image(img_path[0])
seg_sitk = read_sitk_image(seg_path[0])
img_array = sitk_image_to_array(img_sitk)
seg_array = sitk_image_to_array(seg_sitk)
print(f"image shape: {img_array.shape}")
print(f"segmentation shape: {seg_array.shape}")

image shape: (299, 224, 224)
segmentation shape: (299, 224, 224)


In [57]:
import ast
weights_mask = np.zeros_like(seg_array)
for node_center, importance in zip(nodes_coordinates, nodes_importance):
    node_center = np.array(ast.literal_eval(node_center)).astype(int)[::-1]  # reverse to z,y,x
    print(f"node center: {node_center}, intensity value: {seg_array[node_center[0], node_center[1], node_center[2]]}")
    label = seg_array[node_center[0], node_center[1], node_center[2]]
    weights_mask[seg_array == label] += importance
weights_sitk = array_to_sitk_image(weights_mask, seg_sitk)
save_dir = Path("mil_attention_visualization") / f"patient_{patient_id}"
if not save_dir.exists():
    os.makedirs(save_dir)
sitk.WriteImage(weights_sitk, save_dir / "patient_nodes_attention_weights.nii.gz")
sitk.WriteImage(seg_sitk, save_dir / "patient_nodes_segmentation.nii.gz")
sitk.WriteImage(img_sitk, save_dir / "patient_image.nii.gz")

node center: [245 128 104], intensity value: 1.0
node center: [252 129 121], intensity value: 2.0
node center: [238 118  87], intensity value: 3.0
node center: [240 119 129], intensity value: 4.0
node center: [236 133  92], intensity value: 5.0
node center: [226 118 122], intensity value: 6.0
node center: [219 117  88], intensity value: 7.0
node center: [225 119  91], intensity value: 8.0
node center: [218 111 111], intensity value: 9.0
node center: [214 116 124], intensity value: 11.0
node center: [213 106 114], intensity value: 12.0
node center: [214 108 120], intensity value: 16.0
